# Raw PDF EDA
High-level look at the full scraped texts before any section extraction.

In [ ]:
import os, re, glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.feature_extraction import text as sk_text
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

RAW_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'scraped', 'raw'))
files   = sorted(glob.glob(os.path.join(RAW_DIR, '*_raw.txt')))
print(f'Found {len(files)} raw files in {RAW_DIR}')

COUNTRY_DISPLAY = {
    'albania': 'Albania', 'bosnia_and_herzegovina': 'Bosnia & Herzegovina',
    'kosovo': 'Kosovo', 'montenegro': 'Montenegro',
    'north_macedonia': 'North Macedonia', 'serbia': 'Serbia', 'turkey': 'Turkey',
}

records = []
for fpath in files:
    parts    = os.path.basename(fpath).replace('_raw.txt','').split('_')
    year_idx = next((i for i,p in enumerate(parts) if re.fullmatch(r'\d{4}',p)), None)
    if year_idx is None: continue
    country  = '_'.join(parts[:year_idx])
    year     = parts[year_idx]
    text     = open(fpath, encoding='utf-8').read()
    # strip page markers and normalise
    clean    = re.sub(r'={5} PAGE \d+ ={5}', ' ', text)
    clean    = re.sub(r'\s+', ' ', clean).strip().lower()
    records.append(dict(
        country=country, year=year,
        label=COUNTRY_DISPLAY.get(country, country.replace('_',' ').title()) + ' ' + year,
        country_label=COUNTRY_DISPLAY.get(country, country.replace('_',' ').title()),
        pages=len(re.findall(r'={5} PAGE', text)),
        words=len(clean.split()), text=clean,
    ))

df = pd.DataFrame(records).sort_values(['country','year']).reset_index(drop=True)
COUNTRIES = sorted(df['country'].unique())
CMAP   = plt.cm.tab10
COLORS = {c: CMAP(i/max(len(COUNTRIES)-1,1)) for i,c in enumerate(COUNTRIES)}

print(f'Countries : {COUNTRIES}')
print(f'Years     : {sorted(df["year"].unique())}')
display(df[['country_label','year','pages','words']])

## 1. Page & Word Counts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, col, title in zip(axes, ['words','pages'], ['Word count','Page count']):
    for country in COUNTRIES:
        sub = df[df['country']==country].sort_values('year')
        clabel = COUNTRY_DISPLAY.get(country, country)
        ax.plot(sub['year'], sub[col], 'o-', color=COLORS[country],
                linewidth=2, markersize=7, label=clabel)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Year'); ax.legend(fontsize=8, ncol=2)
    ax.grid(True, linestyle=':', alpha=0.5)
plt.suptitle('Full Report Size Over Time', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

## 2. Section Header Map

Where do numbered headers (1.1, 1.2, 1.3, 2., etc.) appear? Helps verify that the PDFs were extracted correctly and shows the document structure.

In [ ]:
PAGE_PAT   = re.compile(r'={5} PAGE (\d+) ={5}')
HEADER_PAT = re.compile(r'^\s*(\d+\.\d*)\.?\s+([A-Z][A-Z &/-]{3,})', re.MULTILINE)

rows = []
for _, r in df.iterrows():
    raw_text = open(
        os.path.join(RAW_DIR, f"{r['country']}_{r['year']}_raw.txt"), encoding='utf-8'
    ).read()
    # char position → page number
    page_map = [(m.start(), int(m.group(1))) for m in PAGE_PAT.finditer(raw_text)]
    def page_at(pos):
        pg = 1
        for ps, pn in page_map:
            if ps <= pos: pg = pn
            else: break
        return pg
    for m in HEADER_PAT.finditer(raw_text):
        rows.append(dict(country_label=r['country_label'], year=r['year'],
                         num=m.group(1), title=m.group(2).strip(),
                         page=page_at(m.start())))

df_h = pd.DataFrame(rows)
# Show only section-1.x headers
df_1x = df_h[df_h['num'].str.match(r'^1\.\d')]
print('Section 1.x headers found across corpus:')
display(df_1x.sort_values(['country_label','year','num']))

print('\nUnique 1.2 title variants:')
for t in sorted(df_h[df_h['num']=='1.2']['title'].unique()):
    print(' ', t)

## 3. Serbia vs Kosovo Progress Language

In [ ]:
PROGRESS_SCALE = {
    'backsliding': -3,
    'no progress': -2,
    'limited progress': -1,
    'some progress': 1,
    'good progress': 2,
    'very good progress': 3,
}

def count_kw(text, phrases):
    return sum(len(re.findall(re.escape(p), text, re.IGNORECASE)) for p in phrases)

focus = ['kosovo', 'serbia']
driver_terms = {
    'Dialogue/Normalisation': ['normalisation', 'dialogue', 'relations', 'agreement', 'roadmap'],
    'Sanctions/CFSP': ['sanctions', 'restrictive measures', 'foreign policy', 'cfsp', 'russia', 'ukraine'],
    'Rule of Law': ['rule of law', 'judiciary', 'corruption', 'anti-corruption', 'organised crime'],
    'Democracy/Governance': ['elections', 'parliament', 'democracy', 'public administration', 'civil society'],
    'Economy': ['economic', 'fiscal', 'investment', 'market', 'competitiveness'],
}

rows = []
for _, row in df[df['country'].isin(focus)].iterrows():
    rec = {
        'country': row['country'],
        'country_label': row['country_label'],
        'year': int(row['year']),
        'label': row['label'],
    }
    total_progress_mentions = 0
    weighted_score = 0
    for phrase, weight in PROGRESS_SCALE.items():
        c = count_kw(row['text'], [phrase])
        rec[phrase] = c
        total_progress_mentions += c
        weighted_score += c * weight
    rec['progress_mentions'] = total_progress_mentions
    rec['progress_score_norm'] = round(weighted_score / total_progress_mentions, 3) if total_progress_mentions else 0
    rec['positive_mentions'] = rec['some progress'] + rec['good progress'] + rec['very good progress']
    rec['negative_mentions'] = rec['backsliding'] + rec['no progress'] + rec['limited progress']
    for driver, terms in driver_terms.items():
        rec[driver] = count_kw(row['text'], terms)
    rows.append(rec)

focus_kw = pd.DataFrame(rows).sort_values(['country', 'year']).reset_index(drop=True)
driver_cols = list(driver_terms.keys())

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

ax = axes[0, 0]
for country in focus:
    sub = focus_kw[focus_kw['country'] == country].sort_values('year')
    ax.plot(sub['year'], sub['progress_score_norm'], 'o-', color=COLORS[country],
            linewidth=2.8, markersize=8,
            label=COUNTRY_DISPLAY.get(country, country.title()))
ax.axhline(0, color='grey', linestyle='--', linewidth=0.8)
ax.set_title('Serbia vs Kosovo: normalised progress score', fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Score')
ax.grid(True, linestyle=':', alpha=0.5)
ax.legend()

ax = axes[0, 1]
bar_width = 0.18
years = sorted(focus_kw['year'].unique())
x = np.arange(len(years))
for i, country in enumerate(focus):
    sub = (focus_kw[focus_kw['country'] == country]
           .set_index('year')
           .reindex(years)
           .reset_index())
    ax.bar(x + (i - 0.5) * 2 * bar_width, sub['positive_mentions'], width=bar_width,
           color=COLORS[country], alpha=0.85,
           label=f"{COUNTRY_DISPLAY.get(country, country.title())}: positive")
    ax.bar(x + (i - 0.5) * 2 * bar_width + bar_width, -sub['negative_mentions'], width=bar_width,
           color=COLORS[country], alpha=0.35,
           label=f"{COUNTRY_DISPLAY.get(country, country.title())}: negative")
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(x + bar_width / 2)
ax.set_xticklabels(years)
ax.set_title('Progress language mix', fontweight='bold')
ax.set_ylabel('Positive mentions above zero, negative below zero')
ax.grid(axis='y', linestyle=':', alpha=0.4)
ax.legend(fontsize=8, ncol=2)

ax = axes[1, 0]
kosovo_driver = (focus_kw[focus_kw['country'] == 'kosovo']
                 .sort_values('year')
                 .set_index('year')[driver_cols])
im = ax.imshow(kosovo_driver.values.T, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(kosovo_driver.index)))
ax.set_xticklabels(kosovo_driver.index)
ax.set_yticks(range(len(driver_cols)))
ax.set_yticklabels(driver_cols)
ax.set_title('Kosovo: issue intensity over time', fontweight='bold')
for i in range(len(driver_cols)):
    for j in range(len(kosovo_driver.index)):
        val = int(kosovo_driver.iloc[j, i])
        ax.text(j, i, str(val), ha='center', va='center', fontsize=8,
                color='white' if val > kosovo_driver.values.max() * 0.55 else 'black')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax = axes[1, 1]
kosovo_delta = (focus_kw[focus_kw['country'] == 'kosovo']
                .sort_values('year')
                [['year', 'progress_score_norm'] + driver_cols]
                .set_index('year')
                .diff()
                .dropna())
delta_cols = ['progress_score_norm'] + driver_cols
im2 = ax.imshow(kosovo_delta[delta_cols].T, cmap='RdBu_r', aspect='auto')
ax.set_xticks(range(len(kosovo_delta.index)))
ax.set_xticklabels([f"{y-1}->{y}" for y in kosovo_delta.index])
ax.set_yticks(range(len(delta_cols)))
ax.set_yticklabels(['Progress score delta'] + driver_cols)
ax.set_title('Kosovo: year-over-year change', fontweight='bold')
for i in range(len(delta_cols)):
    for j in range(len(kosovo_delta.index)):
        val = kosovo_delta[delta_cols].iloc[j, i]
        ax.text(j, i, f'{val:+.2f}' if i == 0 else f'{int(val):+d}',
                ha='center', va='center', fontsize=8,
                color='white' if abs(val) > np.nanmax(np.abs(kosovo_delta[delta_cols].values)) * 0.5 else 'black')
plt.colorbar(im2, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle('Raw reports: what drives Serbia vs Kosovo movement?', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

summary_cols = [
    'country_label', 'year', 'progress_score_norm', 'positive_mentions', 'negative_mentions',
    'Dialogue/Normalisation', 'Sanctions/CFSP', 'Rule of Law', 'Democracy/Governance', 'Economy'
]
display(focus_kw[summary_cols].sort_values(['country_label', 'year']))

print('Kosovo year-over-year movement summary:')
kosovo_yoy = (focus_kw[focus_kw['country'] == 'kosovo']
              .sort_values('year')
              [summary_cols[1:]]
              .set_index('year')
              .diff())
for year, row in kosovo_yoy.dropna().iterrows():
    driver_changes = row[driver_cols].sort_values(key=lambda s: s.abs(), ascending=False)
    top_changes = ', '.join([f"{k} {int(v):+d}" for k, v in driver_changes.head(3).items()])
    print(
        f"  {year-1}->{year}: score {row['progress_score_norm']:+.2f}, "
        f"positive {int(row['positive_mentions']):+d}, negative {int(row['negative_mentions']):+d}; "
        f"largest driver shifts: {top_changes}"
    )

kosovo_peak_sanctions = focus_kw.loc[focus_kw['Sanctions/CFSP'].idxmax()]
print(
    f"\nHighest Kosovo sanctions/CFSP salience: {int(kosovo_peak_sanctions['year'])} "
    f"({int(kosovo_peak_sanctions['Sanctions/CFSP'])} mentions, score {kosovo_peak_sanctions['progress_score_norm']:+.2f})."
)


## 4. Top Bigrams & Trigrams (TF-IDF, all years pooled per country)

In [ ]:
import warnings; warnings.filterwarnings('ignore')

# ── Stopwords ─────────────────────────────────────────────────────────────────

COUNTRY_NAMES = {
    *{tok for slug in COUNTRIES for tok in slug.split('_')},
    'albanian', 'kosovar', 'serbian', 'montenegrin',
    'bosnian', 'herzegovinian', 'macedonian', 'turkish',
    'serb', 'serbs',
    'türkiye',
    'republika', 'srpska', 'brcko', 'brčko', 'district',
    'canton', 'cantons',                        # BiH administrative units
    'federation', 'republic',                   # BiH entities / generic
    'entity', 'entities', 'countrywide',
    'fragmented',                               # BiH governance descriptor
    'pristina', 'prishtina', 'belgrade', 'beograd', 'podgorica',
    'skopje', 'tirana', 'sarajevo', 'ankara', 'istanbul',
    'greece', 'croatia',                        # neighbouring EU states
    'russia', 'russian', 'ukraine', 'syria', 'cyprus',
}

EU_PROCESS = {
    'eu', 'european', 'commission', 'council', 'parliament', 'union',
    'member', 'states', 'enlargement', 'accession', 'candidate',
    'negotiations', 'negotiation', 'chapter', 'chapters',
    'cluster', 'clusters', 'benchmark', 'benchmarks',
    'screening', 'intergovernmental', 'conference',
    'stabilisation', 'association', 'agreement', 'saa',
    'ipa', 'taiex', 'twinning', 'instrument', 'ipard',
    'acquis', 'communautaire', 'commissioner',
    'opening', 'closing', 'closure', 'provisional',
    'conditionality', 'conditionalities',
    'transposition', 'approximation', 'harmonisation', 'harmonization',
    'harmonise', 'harmonize',
    'directive', 'directives', 'regulation', 'regulations',
    'recommendation', 'recommendations',
    'framework', 'protocol', 'annex',
}

EC_BOILERPLATE = {
    'report', 'reports', 'introduction', 'context',
    'summary', 'findings', 'implementation', 'progress',
    'alignment', 'continued', 'continues', 'remains', 'remain',
    'need', 'needs', 'ensure', 'adopted', 'provided',
    'overall', 'particular', 'areas', 'area', 'measures', 'measure',
    'efforts', 'effort', 'including', 'period', 'reporting',
    'government', 'governments', 'reform', 'reforms',
    'institutional', 'capacity', 'administrative', 'judiciary',
    'legislative', 'executive', 'authority', 'authorities',
    'page', 'final', 'document', 'staff', 'working',
    'com', 'swd', 'sw', 'ew', 'bw', 'iw', 'en', 'rem',  # doc artefacts / abbreviations
    'ibid', 'article', 'paragraph', 'graph',              # 'graph' = PDF table artefact
    'accordance', 'pursuant', 'thereof',
    'country', 'countries', 'level', 'levels',
    'national', 'international', 'regional', 'local',
    'public', 'civil', 'society', 'state',
    'appoint', 'appointed', 'appointment',
    'largely', 'largely valid', 'implemented', 'implemented valid',
    'key', 'key priority', 'opinion', 'opinion key',
    'audio',                                   # PDF extraction artefact
}

# Western Balkans institutional acronyms — specific enough to be stopwords
# (they identify institutions, not substantive policy content)
WB_INSTITUTIONS = {
    # Judicial / rule-of-law bodies
    'hjpc',     # High Judicial and Prosecutorial Council (BiH)
    'hjc',      # High Judicial Council (Serbia/others)
    'hpc',      # High Prosecutorial Council
    'aca',      # Anti-Corruption Agency
    'spak',     # Special Anti-Corruption Prosecutor (Albania)
    'boa',      # Board of Appeal
    'instat',   # National Statistics Institute (Albania)
    'osce',     # Organisation for Security and Co-operation in Europe
    'echr',     # European Court of Human Rights
    'tca',      # Trade and Cooperation Agreement / other TCA
    'csos',     # Civil Society Organisations (plural acronym)
    'cso',      # Civil Society Organisation
    'ipard',    # Instrument for Pre-Accession Assistance Rural Development
    'rem',      # Regulatory Authority for Electronic Media (Serbia)
    'ratel',    # Regulatory Agency for Electronic Communications (Serbia)
    'nis',      # Network and Information Systems
    'par',      # Public Administration Reform
    'wbif',     # Western Balkans Investment Framework
    'seeto',    # South-East Europe Transport Observatory
    'magistrates', 'magistrate',
    'inspectorate', 'inspectorates',
    'ombudsman',
    'prosecutor', 'prosecutors', 'prosecutorial',
    'instat',
}

EC_PROGRESS = {
    'backsliding', 'limited', 'moderately', 'prepared', 'preparation',
    'advanced', 'stage', 'early', 'satisfactory', 'preliminary',
}

YEARS_SET = {str(y) for y in range(2000, 2030)}

MONTHS = {
    'january','february','march','april','may','june',
    'july','august','september','october','november','december',
    'jan','feb','mar','apr','jun','jul','aug','sep','oct','nov','dec',
}

NOISE = {
    'pandemic', 'covid', 'coronavirus',
    'earthquakes', 'earthquake',
    'military', 'presidential',
    'eastern',                    # too generic in this corpus
}

STOPWORDS = list(
    sk_text.ENGLISH_STOP_WORDS
    | COUNTRY_NAMES | EU_PROCESS | EC_BOILERPLATE
    | WB_INSTITUTIONS | EC_PROGRESS
    | YEARS_SET | MONTHS | NOISE
)
print(f'Total stopwords: {len(STOPWORDS)}')

# ── Pool all years per country then run TF-IDF ────────────────────────────────
pooled = (df.groupby(['country','country_label'])['text']
          .apply(' '.join).reset_index())

for n, name in [(2,'Bigrams'), (3,'Trigrams')]:
    vec = TfidfVectorizer(ngram_range=(n,n), stop_words=STOPWORDS,
                          min_df=1, max_df=0.95, max_features=5000,
                          sublinear_tf=True)
    X  = vec.fit_transform(pooled['text'])
    fn = vec.get_feature_names_out()

    nc    = len(pooled)
    ncols = min(3, nc)
    nrows = (nc + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*7, nrows*5), squeeze=False)

    for i, row in enumerate(pooled.itertuples()):
        ax      = axes[i//ncols][i%ncols]
        scores  = X[i].toarray().flatten()
        top_idx = scores.argsort()[::-1][:15]
        ax.barh(fn[top_idx][::-1], scores[top_idx][::-1],
                color=COLORS.get(row.country, 'steelblue'), alpha=0.85)
        ax.set_title(row.country_label, fontweight='bold', fontsize=11)
        ax.set_xlabel('TF-IDF', fontsize=8)
        ax.tick_params(axis='y', labelsize=8)

    for j in range(i+1, nrows*ncols):
        axes.flatten()[j].set_visible(False)

    plt.suptitle(f'Top-15 {name} per Country (all years, full report)',
                 fontsize=13, y=1.01)
    plt.tight_layout(); plt.show()

## 4. LDA Topic Model

Change `K` to explore different numbers of topics.

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation

K = 7   # ← adjust

cv = CountVectorizer(stop_words=STOPWORDS, ngram_range=(1,2),
                     min_df=2, max_df=0.9, max_features=4000)
X_counts  = cv.fit_transform(df['text'])
vocab     = cv.get_feature_names_out()
doc_labels = df['label'].tolist()

lda = LatentDirichletAllocation(n_components=K, max_iter=20,
                                 learning_method='online', random_state=42)
doc_topic = lda.fit_transform(X_counts)
print(f'Docs: {X_counts.shape[0]}  |  Terms: {X_counts.shape[1]}  |  '
      f'Perplexity: {lda.perplexity(X_counts):.1f}')

# ── Top terms per topic ───────────────────────────────────────────────────────
ncols = min(4, K)
nrows = (K + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*5, nrows*5), squeeze=False)
pal = plt.cm.Set2

for t in range(K):
    ax      = axes[t//ncols][t%ncols]
    top_idx = lda.components_[t].argsort()[::-1][:12]
    wts     = lda.components_[t][top_idx]
    wts     = wts / wts.sum()
    ax.barh(vocab[top_idx][::-1], wts[::-1], color=pal(t/K), alpha=0.85)
    ax.set_title(f'Topic {t+1}', fontweight='bold')
    ax.set_xlabel('Weight', fontsize=8)
    ax.tick_params(axis='y', labelsize=8)

for j in range(K, nrows*ncols):
    axes.flatten()[j].set_visible(False)

plt.suptitle(f'LDA k={K} — Top Terms per Topic', fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

# ── Doc-topic heatmap ─────────────────────────────────────────────────────────
tnames = [f'T{i+1}' for i in range(K)]
df_dt  = pd.DataFrame(doc_topic, index=doc_labels, columns=tnames)

fig, ax = plt.subplots(figsize=(K*1.4+1, len(doc_labels)*0.45+1))
im = ax.imshow(df_dt.values, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(K));            ax.set_xticklabels(tnames)
ax.set_yticks(range(len(df_dt)));  ax.set_yticklabels(df_dt.index, fontsize=8)
for i in range(len(df_dt)):
    for j in range(K):
        v = df_dt.values[i,j]
        ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=7,
                color='white' if v > 0.5 else 'black')
plt.colorbar(im, ax=ax, shrink=0.5, label='Topic proportion')
ax.set_title(f'Document–Topic Proportions (k={K})', fontweight='bold')
plt.tight_layout(); plt.show()